#### Analysis of Population Receptive Field mapping properties trend lines across groups

In [ ]:
import os
import csv
import glob
import pickle
import cortex
opj = os.path.join
import numpy as np
import pandas as pd
import seaborn as sns
import cortex.freesurfer
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

In [ ]:
# Directories set up 
project = 'UMCG'
main_path = f'/scratch/hb-EGRET-AAA/projects/{project}/derivatives'
trendline_output = os.path.join(f"{main_path}/analysis/pRFM", 'trendlines')
os.makedirs(trendline_output, exist_ok=True)
group_output = os.path.join(f"{main_path}/analysis/pRFM", 'group')
os.makedirs(group_output, exist_ok=True)

In [ ]:
# Analysis parameters
calculate_trendlines = True 
denoising = 'nordic'
r2 = 0.1
min_ecc = 0.5
max_ecc = 6.5
ecc_bound=(min_ecc, max_ecc)
atlas = 'manual'
tasks = ['RET']
min_pRFsize = 0.12
n_samples = 1000
offset = True 

if atlas == "manual":
    rois_list = np.array([['V1', 'V2', 'V3', 'V4','LO'], [1, 2, 3, 4,5]])
    visual_areas = ['V1', 'V2', 'V3', 'V4', 'LO']
else: 
    rois_list = np.array([['V1', 'V2', 'V3', 'V4','LO1', 'LO2'], [1, 2, 3, 4,7,8]])
    visual_areas = ['V1', 'V2', 'V3', 'V4', 'LO1', 'LO2']

glaucoma = ['sub-02', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08','sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20'] # Missing 08 and 19
controls = ['sub-22', 'sub-25', 'sub-26', 'sub-27', 'sub-28', 'sub-29', 'sub-31', 'sub-32',  'sub-34', 'sub-35', 'sub-36', 'sub-37', 'sub-39', 'sub-40', 'sub-41', 'sub-42','sub-43', 'sub-44','sub-45','sub-46'] # List of control subjects
subjects = glaucoma + controls

# Load clinical and demographic data from CSV
# The CSV must have columns: Subject, OCT Average, HFA Average, HFA Best, OCT Best,
# OCT Worst, HFA Worst, HFA 7degrees, Age, Gender
clinical_csv = os.path.join(main_path, 'analysis/pRFM/clinical_data.csv')
clinical_df = pd.read_csv(clinical_csv).set_index('Subject')
oct_averageeye  = clinical_df['OCT Average'].to_dict()
hfa_averageeye  = clinical_df['HFA Average'].to_dict()
hfa_besteye     = clinical_df['HFA Best'].to_dict()
oct_besteye     = clinical_df['OCT Best'].to_dict()
oct_worsteye    = clinical_df['OCT Worst'].to_dict()
hfa_worsteye    = clinical_df['HFA Worst'].to_dict()
hfa_7deg_avg    = clinical_df['HFA 7degrees'].to_dict()
age             = clinical_df['Age'].to_dict()
gender          = clinical_df['Gender'].to_dict()




##### Functions

In [ ]:
class PRFModel:
    '''Class representing pRF model parameters '''
    def __init__(self, r2, size, ecc, angle):
        """
        Initialize the PRFModel instance with the following parameters:
            
        Parameters:
        r2 (float): Model fit value (R-squared). Represents how well the model explains the variance in the observed data. 
                    A higher value indicates a better fit.
        size (float): Size of the pRF (population receptive field), which describes the spatial extent or diameter of the receptive field.
        ecc (float): Eccentricity. Represents the distance of the pRF from the center of the visual field. Higher values mean the receptive field 
                    is located farther from the center (fovea).
        angle (float): Polar angle. Specifies the direction of the receptive field from the center, often in degrees or radians.
        """
        self.r2 = r2            # Model fit - represents how well the model explains data variance
        self.size = size        # Size of the pRF (population receptive field)
        self.ecc = ecc          # Eccentricity - distance of the pRF from the center
        self.angle = angle      # Polar angle - position angle of the pRF

def load_pickle_file(filepath):
    '''Function to load a pickle file given a file path'''
    # Check if the given file path points to an existing file
    if not os.path.isfile(filepath):
        # If the file does not exist, raise a FileNotFoundError
        #raise FileNotFoundError(f"File not found: {filepath}")
        print(f"File not found: {filepath}")            
        return None
    # Open the file in 'rb' mode (read binary) since pickle files contain serialized binary data
    with open(filepath, 'rb') as file: 
        return pickle.load(file)

def load_prf_params(subjectid, main_path, atlas, denoising, task):
    # Find the pkl file in pRFM directory 
    filepath = os.path.join(main_path, f"pRFM/{subjectid}/ses-02/{denoising}/model-{atlas}-nelder-mead-GM_desc-prf_params_{task}.pkl")
    # Load the pickle (assumes you have a helper called load_pickle_file)
    pkl_data = load_pickle_file(filepath)
    if pkl_data is None: # If case there is no file found skip the subject
        return None, None
    # Extract pRF parameters and ROI mask
    prf_params = pkl_data["model"].iterative_search_params
    rois_mask = pkl_data.get("rois_mask", None) # 
    prf_voxels = np.where(rois_mask == 1)[0] # Get the indices of voxels within the ROI

    return prf_params, prf_voxels

def load_labels(subjectid, main_path, atlas):
    fs_dir = os.path.join(main_path, 'freesurfer')
    # Use only left hemisphere for sub-38 (right hemisphere excluded due to segmentation issue)
    hemi = ('lh', 'rh')
    idx_rois4, idx_vls4 = cortex.freesurfer.get_label(subjectid, label='benson14_varea-0001', fs_dir=fs_dir, hemisphere=hemi)
    idx_rois1, idx_vls1 = cortex.freesurfer.get_label(subjectid, label='benson14_eccen-0001', fs_dir=fs_dir, hemisphere=hemi)
    if atlas == 'manual':
        idx_rois_manual, idx_vls_manual = cortex.freesurfer.get_label(subjectid, label='manualdelin', fs_dir=fs_dir, hemisphere=hemi)
        idx_vls4[idx_rois_manual] = idx_vls_manual

    return idx_rois4, idx_vls4, idx_vls1

def filter_prf_data(prf_params, prf_voxels):
    '''Filter and extract pRF data for the selected voxels'''
    # Construct and return a PRFModel object containing specific parameters of interest
    # - r2, size, eccentricity, and polar angle for the selected voxels.
    return PRFModel(
        r2=prf_params[:, 7],                                                # Extract R2 values from column 7 of prf_params.
        size=prf_params[:, 2],                                              # Extract pRF size values from column 2 of prf_params.
        # Eccentricity is calculated as the Euclidean distance from the center (0,0)
        # This is done using the Pythagorean theorem: ecc = sqrt(x^2 + y^2).
        ecc=np.sqrt(prf_params[:, 1]**2 + prf_params[:, 0]**2),             # Calculate eccentricity 
        # np.arctan2(y, x) computes the angle (in radians) of the vector formed by (x, y) relative to the positive x-axis.
        # The function np.rad2deg() converts the resulting angle to degrees.
        angle=np.rad2deg(np.arctan2(prf_params[:, 1], prf_params[:, 0])))

def bootstrap_weighted_avg(x, y, r2, n_samples, ecc_bound, roi_name, subject_num, offset, group_type=None):
    '''Function to create a bootstrapped weighted average plot'''
    # Set the Seaborn plotting style to 'white' for a clean background
    sns.set_style('white')
    # Assign color based on the group type or ROI name
    colors = {'V1': 'blue', 'V2': 'orange', 'V3': 'green', 'V4': 'pink', 'LO': 'red', 'glaucoma': 'red', 'control': 'blue'}
    # Get the color for the group or ROI; default to 'gray' if group/ROI is not in the dictionary
    group_color = colors.get(group_type, colors.get(roi_name, 'gray'))
    # Scatter plot of the x and y data, with transparency set by the R2 value
    plt.scatter(x, y, alpha=r2, marker='o', s=1, color=group_color)

    # Sort the data to fit trend lines properly 
    sorted_idx = np.argsort(x)
    x, y, r2 = x[sorted_idx], y[sorted_idx], r2[sorted_idx]
    # Generate evenly spaced x values for fitting trendlines within the specified eccentricity bounds
    x_fit = np.linspace(ecc_bound[0], ecc_bound[1], len(x))
    y_fit_samples = np.zeros((n_samples, len(x_fit)))

    # Bootstrapping process: create multiple trend lines based on resampled data
    for i in range(n_samples):
        # Randomly choose indices to create a bootstrap sample
        rnd_idx = np.random.choice(len(x), size=len(x), replace=True)
        # Resample the x, y, and r2 data using the random indices
        res_x, res_y, res_r2 = x[rnd_idx], y[rnd_idx], r2[rnd_idx]
        # Fit a linear model (degree 1 polynomial) to the resampled data, weighted by r2
        # CHOOSE WHETHER YOU WANT TO WEIGHT THE TRENDLINE BY VE OR NOT.  
        # model = np.polyfit(res_x, res_y, deg=1, w=res_r2) # WEIGHTED BY VE
        model = np.polyfit(res_x, res_y, deg=1) # NOT WEIGHTED BY VE
        # Calculate the y-values for the fitted line and store in y_fit_samples
        if offset is True: 
            # standard linear fit with intercept
            # model = np.polyfit(res_x, res_y, deg=1, w=res_r2) # WEIGHTED BY VE 
            model = np.polyfit(res_x, res_y, deg=1) # NOT WEIGHTED BY VE
            y_fit_samples[i, :] = model[0] * x_fit + model[1]
        else: 
            # force offset = 0 (fit through origin)
            num = np.sum(res_r2 * res_x * res_y)      # numerator
            den = np.sum(res_r2 * res_x * res_x)      # denominator
            slope_only = num / den if den != 0 else 0
            y_fit_samples[i, :] = slope_only * x_fit

    # Calculate the mean trend line and the CI bounds from bootstrapped samples
    # MEDIAN TRENDLINE
    y_mean_fit = np.median(y_fit_samples, axis=0)
    # MEAN TREND LINE 
    # y_mean_fit = np.mean(y_fit_samples, axis=0)
    # y_mean_fit = model[0] * x_fit  
    lower_bounds = np.percentile(y_fit_samples, 2.5, axis=0) # 2.5th percentile for lower bound
    upper_bounds = np.percentile(y_fit_samples, 97.5, axis=0) # 97.5th percentile for upper bound
    
    # Fit the final model to the entire dataset to extract trendline parameters (slope and offset)
    if offset is True:
        # final_model = np.polyfit(x, y, deg=1, w=r2) # WEIGHTED BY VE
        final_model = np.polyfit(x, y, deg=1) # NOT WEIGHTED BY VE
        slope, offset = final_model
    else:
        num = np.sum(r2 * x * y)
        den = np.sum(r2 * x * x)
        slope = num / den if den != 0 else 0
        offset = 0.0   # force intercept to 0

    # Plot the trend line based on the mean fit and add a label to the plot
    
    plt.plot(x_fit, y_mean_fit, ls='solid', linewidth=1.5, alpha=1, color=group_color, label=f'{roi_name} (N = {len(r2)}): y = {np.polyfit(x, y, deg=1, w=r2)[0]:.4f}x + {np.polyfit(x, y, deg=1, w=r2)[1]:.4f}')
    # Plot the CI as a shaded area around the trend line
    plt.fill_between(x_fit, lower_bounds, upper_bounds, color='gray', alpha=0.2)
    # Add title, labels, limits, and legend to the plot
    plt.title(f'Trend lines {subject_num}', fontsize=18)
    plt.xlabel('Eccentricity (deg)', fontsize=18) # Label for the x-axis
    plt.ylabel('Population Receptive Field Size (deg)', fontsize=18) # Label for the y-axis
    if project == 'UMCG':
        plt.ylim([0, 4]) # Set y-axis limits to maintain consistent scale
    else: 
        plt.ylim([0, 7] )
    plt.legend(loc='upper left',fontsize=14) # Position the legend in the upper left
    plt.grid(True) # Display a grid on the plot
    plt.tight_layout() # Adjust the padding of the plot for a clean layout

    return x_fit, y_mean_fit, lower_bounds, upper_bounds

def trendlines(subject_num, prf_model, idx_vls4, prf_voxels, rois_list, trendline_output, n_samples, task, atlas, r2, min_ecc, max_ecc, min_pRFsize, ecc_bound, offset, prf_model_alt=None, prf_voxels_alt=None, alt_rois=None):
    """
    Compute and plot subject-level pRF size vs eccentricity trendlines
    for each visual ROI, and return summary statistics for CSV export.
    """
    if task == "RET":
        group = 'Glaucoma' if subject_num in glaucoma else 'Controls'
    else:
        group = 'Glaucoma_MO' if subject_num in glaucoma else 'Controls_SS'
    csv_data = []
    vertex_rows = []

    fig, ax = plt.subplots(figsize=(7, 6))
    for i, roi_name in enumerate(rois_list[0]):
        # Use the alternate (benson) fit for the specified ROIs, if provided
        use_alt = (alt_rois is not None) and (roi_name in alt_rois) and (prf_model_alt is not None)
        current_model = prf_model_alt if use_alt else prf_model
        current_voxels = prf_voxels_alt if use_alt else prf_voxels

        roi_idx = np.where(roi_name == rois_list[0, :])[0]
        roi_vertices = np.array(np.where(idx_vls4 == int(rois_list[1, roi_idx])))[0]
        idx = np.in1d(current_voxels, roi_vertices)
        filtered_params = {'ecc': current_model.ecc[idx], 'angle': current_model.angle[idx], 'size': current_model.size[idx], 'r2': current_model.r2[idx]}
        valid_idx = ((filtered_params['r2'] > r2) & (filtered_params['ecc'] > min_ecc) & (filtered_params['ecc'] < max_ecc) & (filtered_params['size'] > min_pRFsize))
        ecc = filtered_params['ecc'][valid_idx]
        data = filtered_params['size'][valid_idx]
        r2_values = filtered_params['r2'][valid_idx]
        valid_vertices = current_voxels[idx][valid_idx]

        for v, e, a, s, ve in zip(valid_vertices, ecc, filtered_params['angle'][valid_idx], data, r2_values):
            vertex_rows.append([subject_num, roi_name, int(v), float(e), float(a), float(s), float(ve)])

        n_points = len(ecc)
        n_label = "< 100" if n_points < 100 else n_points
        if n_points < 100:
            print(f"Skipping {roi_name} for subject {subject_num}: only {n_points} points (<100)")
            continue

        x_fit, y_mean_fit, lower_bounds, upper_bounds = bootstrap_weighted_avg(x=ecc, y=data, r2=r2_values, n_samples=1000, ecc_bound=ecc_bound, roi_name=roi_name, subject_num=subject_num, offset=offset)
        if offset is True:
            # slope, intercept = np.polyfit(ecc, data, deg=1, w=r2_values) # WEIGHTED BY VE
            slope, intercept = np.polyfit(ecc, data, deg=1) # NOT WEIGHTED BY VE
            plt.plot(ecc, slope*ecc + intercept, '-', rasterized=True)
        else:
            num = np.sum(r2_values * ecc * data)
            den = np.sum(r2_values * ecc * ecc)
            slope = num / den if den != 0 else 0
            intercept = 0.0
        mean_ve = float(np.mean(r2_values))
        size_3p5 = float(slope * 3.5 + intercept)

        valid = x_fit >= min_ecc
        ci_low = float(np.min(lower_bounds[valid]))
        ci_high = float(np.max(upper_bounds[valid]))
        csv_data.append([subject_num, roi_name, intercept, slope, ci_low, ci_high, mean_ve, size_3p5, group, hfa_averageeye.get(subject_num, 'N/A'), oct_averageeye.get(subject_num, 'N/A'), hfa_worsteye.get(subject_num, 'N/A'), oct_worsteye.get(subject_num, 'N/A'), hfa_besteye.get(subject_num, 'N/A'), oct_besteye.get(subject_num, 'N/A'), hfa_7deg_avg.get(subject_num, 'N/A'), age.get(subject_num, 'N/A'), gender.get(subject_num, 'N/A'), n_label])

    plot_path = os.path.join(trendline_output, f'{subject_num}_{atlas}_{denoising}.png')
    plt.tight_layout()
    plt.savefig(plot_path.replace(".pdf", ".png"), dpi=150)
    plt.close(fig)

    return csv_data, vertex_rows

def group_comparison(data, output_dir, visual_areas, group_colors, ecc_bound=ecc_bound, summary_stat='median', filename_suffix=''):
    """Plot group-level pRF size vs eccentricity curves for each visual area."""
    # Create a common eccentricity axis shared by all subjects 
    x_fit = np.linspace(ecc_bound[0], ecc_bound[1], 100) 
    # Create one plot for each visual area 
    for visual_area in visual_areas:
        plt.figure(figsize=(6, 6))

        for group_label, color in group_colors.items(): # Loop over experimental groups
            group_data = data[(data['Group'] == group_label) & (data['Visual Area'] == visual_area)] # Select rows for the current group and visual area

            # subject-level slopes and offsets
            slopes = group_data['Slope'].astype(float).to_numpy() # Collect all the slopes
            offsets = group_data['Offset'].astype(float).to_numpy() # Collect all the offset
            subj_curves = np.array([s * x_fit + o for s, o in zip(slopes, offsets)]) # Generate the indivudal subjects curve
            # y = slope * ecc + offset 
            mask = x_fit <= 7 # Limit the plot to 7 degrees of eccentricity 
            x_plot = x_fit[mask]
            subj_curves = subj_curves[:, mask]
            # Plot all subject-level curves to show shows between-subject variability
            for subj_curve in subj_curves:
                plt.plot(x_plot, subj_curve, color=color, alpha=0.3, linewidth=1, rasterized=True)

            # Compute group median regression line
            mean_slope = np.median(slopes)
            mean_offset = np.median(offsets)
            group_curve = mean_slope * x_fit + mean_offset
            group_curve = group_curve[mask]

            # Only plot the group summary line for controls
            if group_label == 'Controls':
                plt.plot(x_plot, group_curve, color=color, linewidth=3)

            lower_bound = np.percentile(subj_curves, 2.5, axis=0) # Get lower bound of CI
            upper_bound = np.percentile(subj_curves, 97.5, axis=0) # Get upper bound of CI
            # Plot mean and CI for btoh groups 
            plt.plot(x_plot, group_curve, color=color, linewidth=2.5, label=f"{group_label} (N={len(slopes)})", rasterized=True)
            plt.fill_between(x_plot, lower_bound, upper_bound, color=color, alpha=0.1)

        # Table formatting
        plt.title(f'pRF Group Comparison: {visual_area}')
        plt.xlabel("Population Receptive Field Eccentricity (deg)")
        plt.ylabel("Population Receptive Field Size (deg)")
        plt.grid(True)
        plt.legend(fontsize=12, loc='upper left')
        plt.tight_layout()
        plt.xlim(0, 7) 
        plt.ylim(0, 4) 
        plt.xticks(np.arange(0, 6.5, 1)) # Ticket to help reading the table 
        plt.yticks(np.arange(0, 4.5, 1))

        # Save the figure
        filename = f"{visual_area}_{filename_suffix}_{atlas}.png" if filename_suffix else f"{visual_area}_{atlas}.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300)
        plt.close()

##### Main Script

In [ ]:
# Calculate subject trendlines 
if calculate_trendlines is True:
    def process_subject_trendlines_OLD(subjectid, task):
        prf_params, prf_voxels = load_prf_params(subjectid, main_path, atlas, denoising, task) # Load pRF parameters for the subject
        if prf_params is None or prf_voxels is None: # Check if we have everythign we need 
            print(f"Skipping {subjectid}, {task} due to missing data.")
            return [], []
        
        # Force pycortex directory to point to the derivatives folder of that subject (chatGPT forced)
        pycortex = os.path.join(f'/scratch/hb-EGRET-AAA/projects/UMCG/derivatives/pycortex')
        cortex.database.default_filestore = pycortex
        cortex.database.db = cortex.database.Database(pycortex)
        if hasattr(cortex, 'db'):
            cortex.db = cortex.database.db
        idx_rois4, idx_vls4, _ = load_labels(subjectid, main_path, atlas) # Load the labels 
        prf_model = filter_prf_data(prf_params, prf_voxels) # Filter the data based on threshold decided 
        # Store the subject data
        subject_csv_data, subject_vertex_data = trendlines(subjectid, prf_model, idx_vls4, prf_voxels, rois_list, trendline_output, n_samples, task, atlas,r2, min_ecc, max_ecc, min_pRFsize, ecc_bound, offset) 
        return subject_csv_data, subject_vertex_data

    def process_subject_trendlines(subjectid, task):
        prf_params, prf_voxels = load_prf_params(subjectid, main_path, atlas, denoising, task)
        if prf_params is None or prf_voxels is None:
            print(f"Skipping {subjectid}, {task} due to missing data.")
            return [], []

        # For sub-05 and sub-08, also load the benson-atlas fit; used only for V4 and LO
        prf_model_alt, prf_voxels_alt, alt_rois = None, None, None
        if subjectid in []:
            prf_params_alt, prf_voxels_alt = load_prf_params(subjectid, main_path, 'benson', denoising, task)
            if prf_params_alt is not None:
                prf_model_alt = filter_prf_data(prf_params_alt, prf_voxels_alt)
                alt_rois = ['LO']
            else:
                print(f"Benson pRF fit not found for {subjectid}, falling back to manual atlas for all ROIs.")

        pycortex = os.path.join(f'/scratch/hb-EGRET-AAA/projects/UMCG/derivatives/pycortex')
        cortex.database.default_filestore = pycortex
        cortex.database.db = cortex.database.Database(pycortex)
        if hasattr(cortex, 'db'):
            cortex.db = cortex.database.db
        idx_rois4, idx_vls4, _ = load_labels(subjectid, main_path, atlas)
        prf_model = filter_prf_data(prf_params, prf_voxels)

        subject_csv_data, subject_vertex_data = trendlines(
            subjectid, prf_model, idx_vls4, prf_voxels, rois_list, trendline_output,
            n_samples, task, atlas, r2, min_ecc, max_ecc, min_pRFsize, ecc_bound, offset,
            prf_model_alt=prf_model_alt, prf_voxels_alt=prf_voxels_alt, alt_rois=alt_rois
        )
        return subject_csv_data, subject_vertex_data

    results = Parallel(n_jobs=12, verbose=5)(delayed(process_subject_trendlines)(subjectid, task)for subjectid in subjects for task in tasks)
    # Separate subject-level rows and vertex-level rows
    all_trendline_data = []
    all_vertex_data = []
    for subj_rows, vertex_rows in results:
        all_trendline_data.extend(subj_rows)
        all_vertex_data.extend(vertex_rows)
    # Data frame to store the results 
    trendlines_df = pd.DataFrame(all_trendline_data, columns=['Subject Number','Visual Area','Offset', 'Slope','Confidence Interval Lower Bound','Confidence Interval Upper Bound', 'VE', 'Size_3p5','Group','HFA Average','OCT Average','HFA Worst', 'OCT Worst','HFA Best','OCT Best','HFA_7degrees','Age','Gender', 'N Points'])
    vertices_df = pd.DataFrame(all_vertex_data, columns=['Subject','Visual Area','Vertex','Eccentricity','Polar Angle','Size','Variance Explained'])
    vertices_csv = os.path.join(trendline_output,f'vertices_{atlas}_{min_ecc}_{max_ecc}.csv')
    vertices_df.to_csv(vertices_csv, index=False)
    trendlines_csv = os.path.join(trendline_output, f'pRFM_Trendlines_{atlas}_{min_ecc}_{max_ecc}.xlsx') # Main output for the delineation 
    trendlines_df.to_excel(trendlines_csv, index=False)
    df = pd.read_excel(trendlines_csv) # Reload

# Plot glaucoma versus healthy controls for group comparison 
group_comparison(data=df[df['Group'].isin(['Controls', 'Glaucoma'])],output_dir=group_output,visual_areas=visual_areas,group_colors={'Controls': 'blue', 'Glaucoma': 'red'},filename_suffix='POAG_vs_HC',summary_stat='median')